# 04 — Model tuning and threshold analysis

This notebook focuses on hyperparameter tuning for XGBoost with imbalance weighting and Random Forest with class weighting. These two approaches were selected based on the results from the previous imbalance-handling experiment. XGBoost achieved the highest mean cross-validated PR-AUC (0.8501), while Random Forest achieved the second-highest PR-AUC (0.8399) and demonstrated particularly high precision (0.9481).

The Logistic Regression approaches were not selected for further tuning because they showed substantially lower overall performance, particularly in terms of F1-score and precision. Therefore, further tuning was concentrated on the two strongest candidate models to make the model development process more focused and computationally efficient.

GridSearchCV with stratified cross-validation is used to systematically evaluate different combinations of hyperparameters, with Average Precision (PR-AUC) used as the primary optimization metric due to the severe class imbalance in the fraud detection dataset.

The test set remains completely untouched during hyperparameter tuning and will only be used for the final evaluation of the selected model.

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from src.data_utils import split_features_target

In [4]:
df = pd.read_csv("../Dataset/creditcard_cleaned.csv")
X, y =split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [6]:
# Calculate fraud weight using training data only
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Random Forest
forest_search = RandomizedSearchCV(
    estimator = RandomForestClassifier(
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1),
    param_distributions={
        'n_estimators': [150, 300],
        'max_depth': [None, 8, 14],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4],
        'max_features': ['sqrt', 'log2'],
    },
    n_iter=8,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    refit=True
)

# XGBoost
xgb_search = RandomizedSearchCV(
    estimator = XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight= fraud_weight,
        random_state=42,
        n_jobs=-1
    ),
    param_distributions={
        'n_estimators': [150, 300],
        'max_depth': [3, 5],
        'learning_rate': [0.03, 0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'min_child_weight': [1, 5],
    },
    n_iter=8,
    scoring= 'average_precision',
    cv = cv,
    n_jobs=-1,
    refit=True
)

# Store two searches
searches = {
    'Random Forest': forest_search,
    'XGBoost': xgb_search
}

# Run hyperparameter tuning
for name, search in searches.items():
    search.fit(X_train, y_train)

    print(
        f'{name}: '
        f'CV PR-AUC = {search.best_score_:.4f}; '
        f'parameters = {search.best_params_}'
    )

Random Forest: CV PR-AUC = 0.8324; parameters = {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 14}
XGBoost: CV PR-AUC = 0.8502; parameters = {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


Hyperparameter tuning did not produce a meaningful improvement for XGBoost, with mean cross-validated PR-AUC increasing marginally from 0.8501 to 0.8502. The tuned Random Forest achieved a lower mean PR-AUC of 0.8324 compared with 0.8399 before tuning. Therefore, XGBoost with imbalance weighting remains the strongest candidate based on cross-validated PR-AUC. The selected XGBoost model will now be evaluated on the previously untouched test set to obtain an unbiased estimate of its generalization performance.

In [9]:
print(xgb_search.best_estimator_)
joblib.dump(
    xgb_search.best_estimator_,
    "../Models/xgboost_best.pkl"
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)


['../Models/xgboost_best.pkl']